In [13]:
# set the matplotlib backend so figures can be saved in the background
import matplotlib
matplotlib.use("Agg")

In [14]:
# import the necessary packages
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import adam_v2
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.utils import to_categorical
from imutils import paths
import matplotlib.pyplot as plt
import numpy as np
import argparse
import random
import cv2
import os
import sys
import h5py
from keras.models import model_from_json  
sys.path.append('..')
#from lenet import LeNet

In [15]:
# import the necessary packages
from keras.models import load_model
import numpy as np
import random
import argparse
from imutils import paths
import cv2
import h5py
from keras.models import model_from_json 
from keras.callbacks import ModelCheckpoint

In [16]:
from keras.models import Sequential
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers.core import Activation
from keras.layers.core import Flatten
from keras.layers.core import Dense
from keras.layers.core import Dropout
from keras.layers.normalization.batch_normalization_v1 import BatchNormalization
from keras import backend as K

In [17]:
from sklearn.model_selection import StratifiedKFold,KFold
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle
from sklearn.metrics import r2_score

In [18]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [19]:
class LeNet:
    @staticmethod
    def build(width, height, depth):
        K.clear_session() 
        # initialize the model
        model = Sequential()
        inputShape = (height, width, depth)
        # if we are using "channels last", update the input shape
        if K.image_data_format() == "channels_first":   #for tensorflow
            inputShape = (depth, height, width)
        # first set of CONV => RELU => POOL layers
        model.add(Conv2D(32, (5, 5),padding="same",input_shape=inputShape))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()        
        #second set of CONV => RELU => POOL layers
        model.add(Conv2D(64, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
        #third set of CONV => RELU => POOL layers
        model.add(Conv2D(128, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
        #fourth set of CONV => RELU => POOL layers
        model.add(Conv2D(256, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
        #fifth set of CONV => RELU => POOL layers() 128 512
        model.add(Conv2D(128, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        BatchNormalization()
        #sixth set of CONV => RELU => POOL layers
        #model.add(Conv2D(64, (5, 5), padding="same"))
        #model.add(Activation("relu"))
        #model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
    
        # first (and only) set of FC => RELU layers
        model.add(Flatten())
        model.add(Dense(500))#500
        model.add(Activation("relu"))
        model.add(Dropout(0.5))#0.5
        
        model.add(Dense(128))
        model.add(Activation("relu"))
        model.add(Dropout(0.3))#0.25
        
        model.add(Dense(64))
        model.add(Activation("relu"))
        
        
        model.add(Dense(32))
        model.add(Activation("relu"))
        
        model.add(Dense(16))
        model.add(Activation("relu"))

        model.add(Dense(1))
        model.add(Activation("linear"))

        # return the constructed network architecture
        return model

In [20]:
# initialize the number of epochs to train for, initial learning rate,
# and batch size
seed=40
EPOCHS = 30
INIT_LR = 1e-3
BS = 8#48
#CLASS_NUM = 29
norm_size = 448

In [21]:
def load_data2(path):
    print("[INFO] loading images...")
    data = []
    labels = []
    # grab the image paths and randomly shuffle them
    imagePaths = sorted(list(paths.list_images(path)))
    random.seed(42)
    random.shuffle(imagePaths)
    # loop over the input images
    for imagePath in imagePaths:
        # load the image, pre-process it, and store it in the data list
        image = cv2.imread(imagePath)
        image = cv2.resize(image, (norm_size, norm_size))
        image = img_to_array(image)
        data.append(image)

        # extract the class label from the image path and update the
        # labels list
        label = float(imagePath.split(os.path.sep)[-2])       
        labels.append(label)  
        
    # scale the raw pixel intensities to the range [0, 1]
    X = np.array(data, dtype="float") / 255.0 
    #data = np.array(data, dtype="double")
    Y = np.array(labels)
    #print(Y)


    # partition the data into training and testing splits using 75% of
    # the data for training and the remaining 25% for testing
    #(X[train], X[test], Y[train], Y[test]) = train_test_split(data,labels, test_size=0.25, random_state=50)

    # convert the labels from integers to vectors
    #trainY = to_categorical(trainY, num_classes=CLASS_NUM)
    #testY = to_categorical(testY, num_classes=CLASS_NUM)   
    return X,Y

In [22]:
# In[*] shuffle in sklearn
def shuffle_skl(X,Y):
    X,Y = shuffle(X,Y, random_state=1337) 
#    print(X,Y)
    return X,Y

In [23]:
#def train(aug,trainX,trainY,testX,testY,args):
def train(aug,X,Y):
    
    for train, test in kfold.split(X, Y):
        # initialize the model
        print("[INFO] compiling model...")
        model = LeNet.build(width=norm_size, height=norm_size, depth=3)
        opt = adam_v2.Adam(learning_rate=INIT_LR, decay=INIT_LR / EPOCHS)
    
        #model.compile(optimizer='rmsprop',loss='mean_squared_error',metrics=['mae', 'acc'])
        model.compile(optimizer=opt,loss='mean_squared_error',metrics=['mae'])


        #reset_keras() 
        # train the network
        print("[INFO] loading network...")
        # load model (can train directly or train on the previous model)
        model = load_model('model_re_layer5_cycle12_9min_lsm_scale401_resize448_all.hdf5')
        print("[INFO] training network...")
        model_checkpoint = ModelCheckpoint('model_re_layer5_cycle12_9min_lsm_scale401_resize448_all.hdf5', monitor='val_mae',verbose=1, save_best_only=True)
        H = model.fit(aug.flow(X[train], Y[train], batch_size=BS),
        validation_data=(X[test], Y[test]), steps_per_epoch=len(X[train]) // BS,
        epochs=EPOCHS, verbose=1, shuffle=True, callbacks=[model_checkpoint])
    


In [24]:
if __name__=='__main__':
    #args = args_parse()
    X,Y = load_data2('Z:\\train_keras\\confocal\\cycle12_401\\norm9')
    #shuffle data
    X,Y = shuffle_skl(X,Y)
    # construct the image generator for data augmentation
    aug = ImageDataGenerator(rotation_range=30, width_shift_range=0,
        height_shift_range=0, shear_range=0.1, zoom_range=0,
        horizontal_flip=True, fill_mode="nearest")
    # define 10-fold cross validation test harness
    kfold = KFold(n_splits=10, shuffle=True, random_state=seed)
    cvscores = []
    
    #train(aug,trainX,trainY,testX,testY,args)
    train(aug,X,Y)
    print("[INFO] serializing network...")

[INFO] loading images...
[INFO] compiling model...
[INFO] loading network...
[INFO] training network...
Epoch 1/30
475/475 [==============================] - ETA: 0s - loss: 1.9760 - mae: 1.0784
Epoch 00001: val_mae improved from inf to 1.04447, saving model to model_re_layer5_cycle12_9min_lsm_scale401_resize448_all.hdf5
475/475 [==============================] - 170s 357ms/step - loss: 1.9760 - mae: 1.0784 - val_loss: 1.8769 - val_mae: 1.0445
Epoch 2/30
475/475 [==============================] - ETA: 0s - loss: 1.1356 - mae: 0.8012
Epoch 00002: val_mae did not improve from 1.04447
475/475 [==============================] - 170s 358ms/step - loss: 1.1356 - mae: 0.8012 - val_loss: 2.2713 - val_mae: 1.2619
Epoch 3/30
475/475 [==============================] - ETA: 0s - loss: 0.8630 - mae: 0.7095
Epoch 00003: val_mae did not improve from 1.04447
475/475 [==============================] - 169s 356ms/step - loss: 0.8630 - mae: 0.7095 - val_loss: 2.1466 - val_mae: 1.1876
Epoch 4/30
475/475 [

Epoch 27/30
476/476 [==============================] - ETA: 0s - loss: 0.0907 - mae: 0.2249
Epoch 00027: val_mae did not improve from 0.27586
476/476 [==============================] - 122s 257ms/step - loss: 0.0907 - mae: 0.2249 - val_loss: 0.2754 - val_mae: 0.3799
Epoch 28/30
476/476 [==============================] - ETA: 0s - loss: 0.0972 - mae: 0.2255
Epoch 00028: val_mae did not improve from 0.27586
476/476 [==============================] - 119s 249ms/step - loss: 0.0972 - mae: 0.2255 - val_loss: 0.1791 - val_mae: 0.3061
Epoch 29/30
476/476 [==============================] - ETA: 0s - loss: 0.0832 - mae: 0.2167
Epoch 00029: val_mae did not improve from 0.27586
476/476 [==============================] - 118s 248ms/step - loss: 0.0832 - mae: 0.2167 - val_loss: 0.1733 - val_mae: 0.2991
Epoch 30/30
476/476 [==============================] - ETA: 0s - loss: 0.0789 - mae: 0.2090
Epoch 00030: val_mae did not improve from 0.27586
476/476 [==============================] - 118s 249ms/ste

Epoch 26/30
476/476 [==============================] - ETA: 0s - loss: 0.0721 - mae: 0.1970
Epoch 00026: val_mae did not improve from 0.21930
476/476 [==============================] - 129s 271ms/step - loss: 0.0721 - mae: 0.1970 - val_loss: 0.1210 - val_mae: 0.2594
Epoch 27/30
476/476 [==============================] - ETA: 0s - loss: 0.0638 - mae: 0.1863
Epoch 00027: val_mae did not improve from 0.21930
476/476 [==============================] - 128s 270ms/step - loss: 0.0638 - mae: 0.1863 - val_loss: 0.1081 - val_mae: 0.2463
Epoch 28/30
476/476 [==============================] - ETA: 0s - loss: 0.0625 - mae: 0.1827
Epoch 00028: val_mae improved from 0.21930 to 0.18808, saving model to model_re_layer5_cycle12_9min_lsm_scale401_resize448_all.hdf5
476/476 [==============================] - 129s 272ms/step - loss: 0.0625 - mae: 0.1827 - val_loss: 0.0651 - val_mae: 0.1881
Epoch 29/30
476/476 [==============================] - ETA: 0s - loss: 0.0637 - mae: 0.1854
Epoch 00029: val_mae did 

Epoch 25/30
476/476 [==============================] - ETA: 0s - loss: 0.0440 - mae: 0.1512
Epoch 00025: val_mae did not improve from 0.19210
476/476 [==============================] - 130s 272ms/step - loss: 0.0440 - mae: 0.1512 - val_loss: 0.0986 - val_mae: 0.2202
Epoch 26/30
476/476 [==============================] - ETA: 0s - loss: 0.0466 - mae: 0.1545
Epoch 00026: val_mae did not improve from 0.19210
476/476 [==============================] - 130s 272ms/step - loss: 0.0466 - mae: 0.1545 - val_loss: 0.1025 - val_mae: 0.2203
Epoch 27/30
476/476 [==============================] - ETA: 0s - loss: 0.1632 - mae: 0.2407
Epoch 00027: val_mae did not improve from 0.19210
476/476 [==============================] - 139s 291ms/step - loss: 0.1632 - mae: 0.2407 - val_loss: 0.1059 - val_mae: 0.2313
Epoch 28/30
476/476 [==============================] - ETA: 0s - loss: 0.0527 - mae: 0.1651
Epoch 00028: val_mae did not improve from 0.19210
476/476 [==============================] - 144s 302ms/ste

Epoch 24/30
476/476 [==============================] - ETA: 0s - loss: 0.0329 - mae: 0.1295
Epoch 00024: val_mae did not improve from 0.19214
476/476 [==============================] - 135s 283ms/step - loss: 0.0329 - mae: 0.1295 - val_loss: 0.1208 - val_mae: 0.2451
Epoch 25/30
476/476 [==============================] - ETA: 0s - loss: 0.0361 - mae: 0.1330
Epoch 00025: val_mae did not improve from 0.19214
476/476 [==============================] - 139s 291ms/step - loss: 0.0361 - mae: 0.1330 - val_loss: 0.0868 - val_mae: 0.2024
Epoch 26/30
476/476 [==============================] - ETA: 0s - loss: 0.0536 - mae: 0.1538
Epoch 00026: val_mae did not improve from 0.19214
476/476 [==============================] - 136s 286ms/step - loss: 0.0536 - mae: 0.1538 - val_loss: 0.1321 - val_mae: 0.2457
Epoch 27/30
476/476 [==============================] - ETA: 0s - loss: 0.0429 - mae: 0.1461
Epoch 00027: val_mae did not improve from 0.19214
476/476 [==============================] - 138s 289ms/ste

Epoch 23/30
476/476 [==============================] - ETA: 0s - loss: 0.0345 - mae: 0.1325
Epoch 00023: val_mae did not improve from 0.17397
476/476 [==============================] - 896s 2s/step - loss: 0.0345 - mae: 0.1325 - val_loss: 0.0885 - val_mae: 0.2105
Epoch 24/30
476/476 [==============================] - ETA: 0s - loss: 0.0355 - mae: 0.1319
Epoch 00024: val_mae did not improve from 0.17397
476/476 [==============================] - 898s 2s/step - loss: 0.0355 - mae: 0.1319 - val_loss: 0.1162 - val_mae: 0.2478
Epoch 25/30
476/476 [==============================] - ETA: 0s - loss: 0.0333 - mae: 0.1277
Epoch 00025: val_mae did not improve from 0.17397
476/476 [==============================] - 896s 2s/step - loss: 0.0333 - mae: 0.1277 - val_loss: 0.0747 - val_mae: 0.1994
Epoch 26/30
476/476 [==============================] - ETA: 0s - loss: 0.0317 - mae: 0.1238
Epoch 00026: val_mae improved from 0.17397 to 0.16100, saving model to model_re_layer5_cycle12_9min_lsm_scale401_res

Epoch 22/30
476/476 [==============================] - ETA: 0s - loss: 0.0297 - mae: 0.1212
Epoch 00022: val_mae did not improve from 0.15747
476/476 [==============================] - 119s 249ms/step - loss: 0.0297 - mae: 0.1212 - val_loss: 0.1233 - val_mae: 0.2431
Epoch 23/30
476/476 [==============================] - ETA: 0s - loss: 0.0261 - mae: 0.1139
Epoch 00023: val_mae did not improve from 0.15747
476/476 [==============================] - 119s 249ms/step - loss: 0.0261 - mae: 0.1139 - val_loss: 0.0869 - val_mae: 0.1978
Epoch 24/30
476/476 [==============================] - ETA: 0s - loss: 0.0280 - mae: 0.1168
Epoch 00024: val_mae did not improve from 0.15747
476/476 [==============================] - 119s 250ms/step - loss: 0.0280 - mae: 0.1168 - val_loss: 0.0968 - val_mae: 0.2040
Epoch 25/30
476/476 [==============================] - ETA: 0s - loss: 0.0300 - mae: 0.1193
Epoch 00025: val_mae did not improve from 0.15747
476/476 [==============================] - 119s 250ms/ste

Epoch 22/30
476/476 [==============================] - ETA: 0s - loss: 0.0239 - mae: 0.1061
Epoch 00022: val_mae did not improve from 0.14508
476/476 [==============================] - 121s 253ms/step - loss: 0.0239 - mae: 0.1061 - val_loss: 0.0904 - val_mae: 0.2043
Epoch 23/30
476/476 [==============================] - ETA: 0s - loss: 0.0279 - mae: 0.1140
Epoch 00023: val_mae did not improve from 0.14508
476/476 [==============================] - 121s 253ms/step - loss: 0.0279 - mae: 0.1140 - val_loss: 0.0433 - val_mae: 0.1461
Epoch 24/30
476/476 [==============================] - ETA: 0s - loss: 0.0255 - mae: 0.1102
Epoch 00024: val_mae did not improve from 0.14508
476/476 [==============================] - 120s 252ms/step - loss: 0.0255 - mae: 0.1102 - val_loss: 0.0589 - val_mae: 0.1578
Epoch 25/30
476/476 [==============================] - ETA: 0s - loss: 0.0225 - mae: 0.1035
Epoch 00025: val_mae did not improve from 0.14508
476/476 [==============================] - 121s 253ms/ste

Epoch 21/30
476/476 [==============================] - ETA: 0s - loss: 0.0392 - mae: 0.1207
Epoch 00021: val_mae did not improve from 0.11009
476/476 [==============================] - 120s 253ms/step - loss: 0.0392 - mae: 0.1207 - val_loss: 0.0810 - val_mae: 0.2004
Epoch 22/30
476/476 [==============================] - ETA: 0s - loss: 0.0256 - mae: 0.1072
Epoch 00022: val_mae did not improve from 0.11009
476/476 [==============================] - 120s 252ms/step - loss: 0.0256 - mae: 0.1072 - val_loss: 0.0697 - val_mae: 0.1756
Epoch 23/30
476/476 [==============================] - ETA: 0s - loss: 0.0212 - mae: 0.0989
Epoch 00023: val_mae did not improve from 0.11009
476/476 [==============================] - 120s 253ms/step - loss: 0.0212 - mae: 0.0989 - val_loss: 0.0452 - val_mae: 0.1394
Epoch 24/30
476/476 [==============================] - ETA: 0s - loss: 0.0200 - mae: 0.0972
Epoch 00024: val_mae did not improve from 0.11009
476/476 [==============================] - 120s 253ms/ste

Epoch 20/30
476/476 [==============================] - ETA: 0s - loss: 0.0211 - mae: 0.0983
Epoch 00020: val_mae did not improve from 0.10765
476/476 [==============================] - 119s 251ms/step - loss: 0.0211 - mae: 0.0983 - val_loss: 0.0505 - val_mae: 0.1636
Epoch 21/30
476/476 [==============================] - ETA: 0s - loss: 0.0201 - mae: 0.0978
Epoch 00021: val_mae did not improve from 0.10765
476/476 [==============================] - 120s 252ms/step - loss: 0.0201 - mae: 0.0978 - val_loss: 0.0453 - val_mae: 0.1411
Epoch 22/30
476/476 [==============================] - ETA: 0s - loss: 0.0202 - mae: 0.0961
Epoch 00022: val_mae did not improve from 0.10765
476/476 [==============================] - 120s 252ms/step - loss: 0.0202 - mae: 0.0961 - val_loss: 0.0666 - val_mae: 0.1664
Epoch 23/30
476/476 [==============================] - ETA: 0s - loss: 0.0164 - mae: 0.0875
Epoch 00023: val_mae did not improve from 0.10765
476/476 [==============================] - 120s 253ms/ste